In [3]:
import pandas as pd
import random
from datetime import datetime, timedelta

# -----------------------------
# Configuration
# -----------------------------
NUM_PATIENTS = 1000
START_DATE = datetime(2026, 1, 1, 8, 0)

random.seed(42)

activities = [
    "Registration",
    "Triage",
    "Doctor Consultation",
    "Lab Test",
    "Radiology",
    "Pharmacy",
    "Billing",
    "Discharge"
]

departments = {
    "Registration": "Front Desk",
    "Triage": "Emergency",
    "Doctor Consultation": "General Medicine",
    "Lab Test": "Laboratory",
    "Radiology": "Radiology",
    "Pharmacy": "Pharmacy",
    "Billing": "Accounts",
    "Discharge": "Reception"
}

rows = []

# -----------------------------
# Generate Clean Data
# -----------------------------
for i in range(1, NUM_PATIENTS + 1):

    case_id = f"C{i:05d}"
    patient_id = f"P{i:05d}"

    current_time = START_DATE + timedelta(
        minutes=random.randint(0, 50000)
    )

    severity = random.choice(
        ["Low", "Medium", "High", "Critical"]
    )

    doctor = f"DOC{random.randint(100,120)}"
    nurse = f"NUR{random.randint(200,220)}"

    for activity in activities:

        current_time += timedelta(
            minutes=random.randint(5,40)
        )

        rows.append({

            "Case_ID": case_id,

            "Patient_ID": patient_id,

            "Activity": activity,

            "Department": departments[activity],

            "Timestamp": current_time.strftime("%Y-%m-%d %H:%M:%S"),

            "Doctor_ID": doctor,

            "Nurse_ID": nurse,

            "Severity": severity,

            "Waiting_Time_Minutes": random.randint(5,40),

            "Status": "Completed"

        })

df = pd.DataFrame(rows)

# ==================================================
# Introduce Data Quality Issues
# ==================================================

# 1. Duplicate Rows
duplicates = df.sample(
    300,
    random_state=1
)

df = pd.concat(
    [df, duplicates],
    ignore_index=True
)

# --------------------------------------------------
# 2. NULL Values
# --------------------------------------------------

for column in ["Doctor_ID", "Department", "Severity"]:

    idx = df.sample(
        120,
        random_state=random.randint(1,1000)
    ).index

    df.loc[idx, column] = None

# --------------------------------------------------
# 3. Inconsistent Activity Names
# --------------------------------------------------

mapping = {

    "Registration": [
        "registration",
        "REGISTRATION"
    ],

    "Doctor Consultation": [
        "Doctor consultation",
        "doctor consultation"
    ],

    "Lab Test": [
        "Lab test",
        "LAB TEST"
    ],

    "Radiology": [
        "X-Ray",
        "radiology"
    ]

}

for original, variants in mapping.items():

    idx = df[df["Activity"] == original].sample(
        80,
        random_state=random.randint(1,1000)
    ).index

    for i, row in enumerate(idx):

        df.at[row, "Activity"] = variants[i % len(variants)]

# --------------------------------------------------
# 4. Negative Waiting Time
# --------------------------------------------------

idx = df.sample(
    80,
    random_state=9
).index

df.loc[idx, "Waiting_Time_Minutes"] = -5

# --------------------------------------------------
# 5. Invalid Status
# --------------------------------------------------

idx = df.sample(
    60,
    random_state=10
).index

df.loc[idx, "Status"] = "Done"

# --------------------------------------------------
# 6. Blank Timestamp
# --------------------------------------------------

idx = df.sample(
    50,
    random_state=11
).index

df.loc[idx, "Timestamp"] = ""

# --------------------------------------------------
# 7. Extra Spaces
# --------------------------------------------------

idx = df.sample(
    100,
    random_state=12
).index

df.loc[idx, "Department"] = (
    " " +
    df.loc[idx, "Department"].fillna("Unknown") +
    " "
)

# --------------------------------------------------
# Save CSV
# --------------------------------------------------

df.to_csv(
    "hospital_event_log_raw_dirty.csv",
    index=False
)

print("="*60)
print("Dataset Created Successfully")
print("="*60)

print("Total Records :", len(df))
print("Unique Patients :", df["Patient_ID"].nunique())

display(df)

Dataset Created Successfully
Total Records : 8300
Unique Patients : 1000


,Case_ID,Patient_ID,Activity,Department,Timestamp,Doctor_ID,Nurse_ID,Severity,Waiting_Time_Minutes,Status
0,C00001,P00001,Registration,Front Desk,2026-01-30 10:45:00,DOC100,NUR208,Low,19,Completed
1,C00001,P00001,Triage,Emergency,2026-01-30 10:58:00,DOC100,NUR208,Low,11,Completed
2,C00001,P00001,Doctor Consultation,General Medicine,2026-01-30 11:37:00,DOC100,NUR208,Low,10,Completed
3,C00001,P00001,Lab Test,Laboratory,2026-01-30 12:09:00,DOC100,NUR208,Low,7,Completed
4,C00001,P00001,Radiology,Radiology,2026-01-30 12:15:00,DOC100,NUR208,Low,10,Completed
...,...,...,...,...,...,...,...,...,...,...
8295,C00035,P00035,Radiology,Radiology,2026-01-01 21:34:00,DOC102,NUR207,Medium,23,Completed
8296,C00844,P00844,Registration,Front Desk,2026-01-26 15:25:00,DOC108,NUR220,Low,16,Completed
8297,C00992,P00992,Triage,Emergency,2026-01-08 17:58:00,DOC118,NUR202,Medium,36,Completed
8298,C00182,P00182,Billing,Accounts,2026-01-12 19:31:00,DOC114,NUR201,Low,16,Completed


In [2]:
display(df)

,Case_ID,Patient_ID,Activity,Department,Timestamp,Doctor_ID,Nurse_ID,Severity,Waiting_Time_Minutes,Status
0,C00001,P00001,Registration,Front Desk,2026-01-30 10:45:00,DOC100,NUR208,Low,19,Completed
1,C00001,P00001,Triage,Emergency,2026-01-30 10:58:00,DOC100,NUR208,Low,11,Completed
2,C00001,P00001,Doctor Consultation,General Medicine,2026-01-30 11:37:00,DOC100,NUR208,Low,10,Completed
3,C00001,P00001,Lab Test,Laboratory,2026-01-30 12:09:00,DOC100,NUR208,Low,7,Completed
4,C00001,P00001,Radiology,Radiology,2026-01-30 12:15:00,DOC100,NUR208,Low,10,Completed
...,...,...,...,...,...,...,...,...,...,...
8295,C00035,P00035,Radiology,Radiology,2026-01-01 21:34:00,DOC102,NUR207,Medium,23,Completed
8296,C00844,P00844,Registration,Front Desk,2026-01-26 15:25:00,DOC108,NUR220,Low,16,Completed
8297,C00992,P00992,Triage,Emergency,2026-01-08 17:58:00,DOC118,NUR202,Medium,36,Completed
8298,C00182,P00182,Billing,Accounts,2026-01-12 19:31:00,DOC114,NUR201,Low,16,Completed
